In [1]:
import scanpy as sc 
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.metrics import adjusted_rand_score
from sklearn.decomposition import PCA

import scipy.sparse as sp 
import warnings

warnings.filterwarnings("ignore")

import os
import ctypes
import sys

# 1. 先设置 R_HOME
os.environ["R_HOME"] = "/home/pxy/miniconda3/envs/r40/lib/R"

# 2. 【核心黑科技】手动加载 R 的动态库
# 这步操作等同于在终端里设置 LD_LIBRARY_PATH，专门解决 VS Code 找不到库的问题
try:
    # 这是 R 的核心库路径
    libR_path = "/home/pxy/miniconda3/envs/r40/lib/R/lib/libR.so"
    # 强制加载进内存
    ctypes.CDLL(libR_path, mode=ctypes.RTLD_GLOBAL)
    print("✅ 成功强制加载 libR.so")
except OSError as e:
    print(f"❌ 加载失败: {e}")

# 3. 然后再导入其他包
sys.path.append("..") 

import spCLUE
import rpy2.robjects as robjects
print("R 环境路径:", robjects.r['R.home']()[0])

spCLUE.fix_seed(0)

# 定义DLPFC数据集的12个切片ID
slice_ids = [
    "151507", "151508", "151509", "151510",
    "151669", "151670", "151671", "151672",
    "151673", "151674", "151675", "151676"
]

# 用于存储每个切片的ARI结果
ari_results = []

# 数据路径（请根据实际情况确认路径是否正确）
data_dir = '/home/pxy/home/pxy/data/DLPFC/st/'

# 【新增】创建保存图片的文件夹
figures_dir = "figures_test"
if not os.path.exists(figures_dir):
    os.makedirs(figures_dir)
    print(f"Created directory: {figures_dir}")

print(f"Start processing {len(slice_ids)} slices...")

for sample_name in slice_ids:
    print(f"\n{'='*20} Processing Sample: {sample_name} {'='*20}")
    
    # 1. 设置簇的数量 (根据DLPFC数据集的已知Ground Truth)
    # 151669-151672 通常只有5层，其他切片为7层
    if sample_name in ["151669", "151670", "151671", "151672"]:
        n_clusters = 5
    else:
        n_clusters = 7
    
    try:
        # 2. 加载数据
        # 使用 read_visium 加载数据，路径拼接逻辑参考原文件
        adata = sc.read_visium(data_dir + sample_name)
        adata.var_names_make_unique()
        
        # 加载元数据 (Ground Truth)
        meta = pd.read_csv(data_dir + sample_name + "/metadata.tsv", sep="\t")
        meta = meta.set_index("barcode")
        adata.obs["Region"] = meta.loc[adata.obs_names, "layer_guess_reordered"]
        
        # 3. 数据预处理与构图
        # 原文件 Cell 6 的逻辑
        # adata = spCLUE.preprocess(adata)
        # adata.obsm["X_pca"] = PCA(n_components=200, random_state=0).fit_transform(adata.X)
        
        # g_spatial = spCLUE.prepare_graph(adata, "spatial", n_neighbors=6)
        # g_expr = spCLUE.prepare_graph(adata, "expr", metric="euclidean", n_neighbors=8)
        from spCLUE.preprocess import (
            prepare_euclidean_graph, 
            prepare_cosine_graph, 
            prepare_fused_graph
        )

        # 1. 数据预处理
        adata = spCLUE.preprocess(adata, hvgNumber=3000)

        # 2. 构建三个视图的图
        adj_s, g_spatial = prepare_euclidean_graph(adata,)
        adj_f, g_feature = prepare_cosine_graph(adata, k=14)
        g_combined = prepare_fused_graph(adj_s, adj_f)

        # 3. 准备图字典
        graph_dict = {
            "spatial": g_spatial,
            "feature": g_feature,
            "combined": g_combined,
        }
        use_zinb = True
        
        # 4. 模型初始化与训练
        # 原文件 Cell 8 的逻辑
        # 注意：这里将 n_clusters 参数改为动态变量，与当前切片保持一致
        # spCLUE_model = spCLUE.spCLUE(adata.obsm["X_pca"], graph_dict, n_clusters,
                                    # )
        # _, adata.obsm["spCLUE"], att_beta = spCLUE_model.train()
        import scipy.sparse as sp

        input_x = adata.X
        if sp.issparse(input_x):
            input_x = input_x.toarray() # 将稀疏矩阵转为普通的 numpy array
        if "raw_count" in graph_dict and sp.issparse(graph_dict["raw_count"]):
            graph_dict["raw_count"] = graph_dict["raw_count"].toarray()
        spCLUE_model = spCLUE.spCLUE(
            input_data=input_x,
            graph_dict=graph_dict,
            n_clusters=n_clusters,
            lambda_ccr=5.0,  # CCR损失权重
            use_zinb=True,  # 是否使用ZINB解码器
            # kappa=0.1,
            beta=1.0,
            gamma=1.0,
        )
        _, adata.obsm["spCLUE"], attention_weights = spCLUE_model.train()
        
        # 5. 聚类
        # 原文件 Cell 10 的逻辑
        refinement = True
        cluster_method = "mclust"
        spCLUE.clustering(
            adata,
            n_clusters,
            key="spCLUE",
            refinement=refinement,
            cluster_methods=cluster_method,
        )
        
        # 6. 计算 ARI
        # 原文件 Cell 12 的逻辑
        # 过滤掉 Ground Truth 为 NA 的区域
        adata_valid = adata[adata.obs.Region.notna()]
        ARI = adjusted_rand_score(adata_valid.obs["Region"], adata_valid.obs["mclust_refined"])
        
        print(f"Sample {sample_name} ARI: {ARI:.8f}")
        ari_results.append(ARI)

        # 绘图：show=False 防止直接显示，便于后续保存
        adata.obs["spCLUE"] = adata.obs["mclust_refined"]
        sc.pl.spatial(
            adata, 
            color=["Region", "spCLUE"], 
            title=["Manual Annotation", f"spCLUE (ARI={round(ARI, 2)})"],
            show=False 
        )
        
        # 保存路径
        save_path = os.path.join(figures_dir, f"{sample_name}.png")
        
        # 保存图片 (bbox_inches='tight' 去除多余白边, dpi=300 保证清晰度)
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        
        # 关闭当前图形，释放内存 (在循环中非常重要，否则内存会爆)
        plt.close()
        
        print(f"Figure saved to: {save_path}")
        
    except Exception as e:
        print(f"Error processing sample {sample_name}: {e}")

# 7. 输出最终统计结果
print(f"\n{'='*20} Final Results {'='*20}")
if ari_results:
    mean_ari = np.mean(ari_results)
    median_ari = np.median(ari_results)
    print(f"ARI per slice: {[round(x, 5) for x in ari_results]}")
    print(f"Mean ARI: {mean_ari:.4f}")
    print(f"Median ARI: {median_ari:.4f}")
else:
    print("No ARI results collected.")

✅ 成功强制加载 libR.so
R 环境路径: /home/pxy/miniconda3/envs/r40/lib/R
Start processing 12 slices...

==================== Processing Sample: 151507 ====================
Preprocessing starting according to CSMVL description...
Selecting top 3000 HVGs...
Normalizing total counts to 10,000...
Scaling and clipping at threshold 10...
Training Start (Enhanced 3-View with CCR) =========================>


 22%|██▏       | 55/250 [00:03<00:07, 26.86it/s]

epoch 50: ARI=0.0736, CCR=0.5000, CLU=2.5650, REC=0.6112
tensor([[ 0.8655,  0.8499,  1.1411,  ...,  6.9170,  6.7984,  6.9034],
        [ 0.8715,  0.8240,  1.1328,  ...,  9.6693,  8.4650,  9.2720],
        [ 0.7427,  0.8475,  1.1024,  ..., 11.2582, 11.4543,  6.4753],
        ...,
        [ 0.7578,  0.8665,  1.0862,  ...,  9.6990,  9.3726,  6.0062],
        [ 0.8373,  0.7834,  1.0986,  ..., 10.7295,  9.6384,  9.1231],
        [ 0.7388,  0.8419,  1.1019,  ..., 11.4920, 11.9907,  6.4921]],
       device='cuda:0', grad_fn=<ClampBackward1>)


 41%|████      | 103/250 [00:04<00:05, 27.07it/s]

epoch 100: ARI=-0.0187, CCR=0.4979, CLU=2.5649, REC=0.5954
tensor([[ 0.8715,  0.9868,  1.0327,  ...,  5.3568,  6.2727,  5.5100],
        [ 0.9665,  0.9498,  1.2050,  ..., 11.2701, 11.1290, 11.1495],
        [ 0.8415,  0.9937,  1.0223,  ...,  7.8377,  7.4992,  7.7439],
        ...,
        [ 0.8466,  0.9949,  1.0219,  ...,  7.3419,  6.9051,  7.2216],
        [ 0.9646,  0.9578,  1.1974,  ..., 10.4626, 11.4507, 10.6208],
        [ 0.8382,  0.9925,  1.0224,  ...,  8.1389,  7.9550,  8.0880]],
       device='cuda:0', grad_fn=<ClampBackward1>)


...

epoch 100: ARI=0.3926, CCR=0.4025, CLU=2.0144, REC=0.6782
tensor([[1.1242, 0.8741, 0.9363,  ..., 1.0110, 1.1224, 1.2192],
        [1.1619, 0.8513, 0.9199,  ..., 1.0102, 1.1546, 1.2627],
        [1.0306, 0.9447, 1.3876,  ..., 1.3017, 1.0281, 1.9322],
        ...,
        [1.0306, 0.9447, 1.3876,  ..., 1.3018, 1.0281, 1.9325],
        [1.0406, 0.9414, 1.3692,  ..., 1.3128, 1.0125, 1.9086],
        [1.0324, 0.9439, 1.3875,  ..., 1.3060, 1.0253, 1.9351]],
       device='cuda:0', grad_fn=<ClampBackward1>)
fitting ...
  |======================================================================| 100%
Sample 151675 ARI: 0.33002865
Figure saved to: figures_test/151675.png

==================== Processing Sample: 151676 ====================
Preprocessing starting according to CSMVL description...
Selecting top 3000 HVGs...
Normalizing total counts to 10,000...
Scaling and clipping at threshold 10...
Training Start (Enhanced 3-View with CCR) =========================>


 22%|██▏       | 56/250 [00:01<00:05, 33.02it/s]

epoch 50: ARI=0.3243, CCR=0.4942, CLU=2.5650, REC=0.7197
tensor([[1.0851, 0.9758, 0.8812,  ..., 1.0804, 1.0009, 1.3663],
        [1.0737, 0.9815, 0.9021,  ..., 1.0649, 0.9950, 1.3198],
        [1.0710, 0.8596, 1.3376,  ..., 0.9067, 1.1304, 1.7774],
        ...,
        [1.0727, 0.8591, 1.3391,  ..., 0.9072, 1.1314, 1.7780],
        [1.0756, 0.8509, 1.3498,  ..., 0.9072, 1.1365, 1.7955],
        [1.0646, 1.0011, 0.9132,  ..., 1.0731, 0.9894, 1.2917]],
       device='cuda:0', grad_fn=<ClampBackward1>)


 40%|███▉      | 99/250 [00:03<00:04, 32.39it/s]


epoch 100: ARI=0.5308, CCR=0.4074, CLU=2.4257, REC=0.7063
tensor([[1.1547, 1.1094, 0.8875,  ..., 1.1034, 1.1001, 1.4083],
        [1.1459, 1.1054, 0.8941,  ..., 1.0975, 1.0949, 1.3858],
        [1.0786, 1.0015, 1.3600,  ..., 1.0077, 1.1258, 1.8270],
        ...,
        [1.0788, 1.0015, 1.3606,  ..., 1.0077, 1.1260, 1.8285],
        [1.0825, 0.9998, 1.3727,  ..., 1.0097, 1.1289, 1.8573],
        [1.1370, 1.1019, 0.9004,  ..., 1.0944, 1.0915, 1.3639]],
       device='cuda:0', grad_fn=<ClampBackward1>)
fitting ...
  |======================================================================| 100%
Sample 151676 ARI: 0.34195952
Figure saved to: figures_test/151676.png

==================== Final Results ====================
ARI per slice: [0.39271, 0.27057, 0.39328, 0.36448, 0.17654, 0.24602, 0.53575, 0.66696, 0.33208, 0.30233, 0.33003, 0.34196]
Mean ARI: 0.3627
Median ARI: 0.3370
